# 📦 Module 1: Review Authenticity Model
**ECommerce Product Trust Assessor**

This notebook trains a Random Forest / XGBoost classifier to detect
inauthentic, spam, or low-quality reviews.

**Dataset:** Amazon Product Reviews (via Hugging Face or Kaggle mirror)

**Output:** `models/authenticity/rf_authenticity.joblib`

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
!pip install -q datasets pandas scikit-learn xgboost shap joblib matplotlib seaborn

In [ ]:
import re, math, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import shap
import joblib

warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Imports complete')

In [ ]:
# ── 2. Load Data ────────────────────────────────────────────────────────────
# Option A: Hugging Face Datasets (easiest for Colab)
from datasets import load_dataset

dataset = load_dataset('McAuley-Lab/Amazon-Reviews-2023', 'raw_review_All_Beauty',
                       split='full', trust_remote_code=True)
df = dataset.to_pandas().sample(50_000, random_state=42)

# Keep relevant columns
df = df[['text', 'rating', 'verified_purchase', 'helpful_vote', 'timestamp']].copy()
df = df.dropna(subset=['text'])
df['text'] = df['text'].astype(str)

print(f'Dataset shape: {df.shape}')
df.head(3)

In [ ]:
# ── 3. Feature Engineering ──────────────────────────────────────────────────

SPAM_PHRASES = [
    'highly recommend', 'five stars', 'best product', 'love it',
    'great quality', 'amazing product', 'perfect', 'must buy',
    'exceeded my expectations', 'dont hesitate'
]

def extract_review_features(row):
    text = str(row.get('text', ''))
    words = text.split()
    lower = text.lower()

    # Length features
    word_count = len(words)
    char_count = len(text)

    # Caps features
    caps_ratio = sum(1 for c in text if c.isupper()) / max(char_count, 1)

    # Spam phrase density
    spam_hits = sum(1 for p in SPAM_PHRASES if p in lower)
    spam_density = spam_hits / max(word_count / 10, 1)

    # Punctuation
    exclamations = text.count('!')

    # Repeated words
    word_counts = Counter(words)
    repeated = sum(1 for v in word_counts.values() if v > 2)
    repeated_ratio = repeated / max(len(word_counts), 1)

    # Helpful votes
    helpful = row.get('helpful_vote', 0) or 0

    return {
        'word_count': word_count,
        'char_count': char_count,
        'caps_ratio': caps_ratio,
        'spam_density': spam_density,
        'exclamation_count': exclamations,
        'repeated_word_ratio': repeated_ratio,
        'helpful_votes': helpful,
        'is_verified': int(bool(row.get('verified_purchase', False))),
        'rating': float(row.get('rating', 3.0))
    }

features_df = df.apply(extract_review_features, axis=1, result_type='expand')
print('Feature shape:', features_df.shape)
features_df.describe()

In [ ]:
# ── 4. Create Pseudo-Labels ─────────────────────────────────────────────────
# Since we lack ground-truth fake labels, we create a proxy:
# SUSPICIOUS = short + spam-heavy + high caps + not verified

def create_authenticity_label(row):
    score = 0
    if row['word_count'] < 10: score += 2
    if row['spam_density'] > 1.0: score += 2
    if row['caps_ratio'] > 0.2: score += 1
    if row['is_verified'] == 0: score += 1
    if row['repeated_word_ratio'] > 0.3: score += 1
    return int(score >= 3)  # 1 = suspicious, 0 = authentic

features_df['is_suspicious'] = features_df.apply(create_authenticity_label, axis=1)

print('Class distribution:')
print(features_df['is_suspicious'].value_counts(normalize=True).round(3))

In [ ]:
# ── 5. Train Models ─────────────────────────────────────────────────────────
FEATURE_COLS = [
    'word_count', 'char_count', 'caps_ratio', 'spam_density',
    'exclamation_count', 'repeated_word_ratio', 'helpful_votes',
    'is_verified', 'rating'
]

X = features_df[FEATURE_COLS].fillna(0)
y = features_df['is_suspicious']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Random Forest ────────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, rf_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, rf_proba):.4f}')

# ── XGBoost ──────────────────────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss', random_state=42
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print('\n=== XGBoost ===')
print(classification_report(y_test, xgb_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, xgb_proba):.4f}')

In [ ]:
# ── 6. SHAP Analysis ────────────────────────────────────────────────────────
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test[:500])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test[:500], feature_names=FEATURE_COLS, show=False)
plt.title('SHAP Feature Importance — Review Authenticity Model')
plt.tight_layout()
plt.savefig('shap_authenticity.png', dpi=150)
plt.show()

In [ ]:
# ── 7. Save Best Model ───────────────────────────────────────────────────────
import os
os.makedirs('models/authenticity', exist_ok=True)

# Save XGBoost (higher AUC in most runs)
joblib.dump(xgb_model, 'models/authenticity/xgb_authenticity.joblib')
joblib.dump(rf, 'models/authenticity/rf_authenticity.joblib')
joblib.dump(FEATURE_COLS, 'models/authenticity/feature_cols.joblib')

print('✅ Models saved to models/authenticity/')

# Feature importances
fi = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print('\nTop features:')
print(fi)